# 7. Changing a primitive — and an approximate adder

Every DRAM command sequence in this tutorial comes from **lime** (notebook 06), which
compiles a small *majority network* per primitive (full adder, AND, …) into the shortest
AAP/AP sequence it can find. Those networks are built into the compiler. The option
`--pud-primitives=<file>` replaces any of them with one written in MLIR, so you can
change what a primitive is — and what it costs — without rebuilding anything.

The mapping (which prices a primitive) and the code generator (which emits it) read
the same file, so the cost model and the program cannot disagree.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # tutorial/, for _infra
from _infra import pud

ART = pud.ARTIFACTS
ART.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile ../artifacts/full_adder.mlir
// The full adder, exactly as built into the compiler.
func.func @row_add(%a: !bits.row<1>, %b: !bits.row<1>, %cin: !bits.row<1> {pud.resident = "DCC1"})
    -> (!bits.row<1>, !bits.row<1> {pud.resident = "DCC1"})
    attributes {pud.commands = 8} {
  %cout  = bits.maj %a, %b, %cin : !bits.row<1>, !bits.row<1>, !bits.row<1> -> !bits.row<1>
  %nb    = bits.not %b : !bits.row<1> -> !bits.row<1>
  %m     = bits.maj %a, %nb, %cin : !bits.row<1>, !bits.row<1>, !bits.row<1> -> !bits.row<1>
  %ncout = bits.not %cout : !bits.row<1> -> !bits.row<1>
  %sum   = bits.maj %b, %ncout, %m : !bits.row<1>, !bits.row<1>, !bits.row<1> -> !bits.row<1>
  return %sum, %cout : !bits.row<1>, !bits.row<1>
}

Running that cell writes the file. It is the full adder exactly as built in, as a
majority network: `bits.maj` is a
three-input majority and `bits.not` a complement. `pud.resident = "DCC1"` keeps the
carry in a dual-contact cell from one bit to the next, and `pud.commands = 8` makes the
compiler refuse the file if the sequence ever takes a different number of commands.
Compiled from the file it gives the same eight commands; only the *source* column
changes:

In [ ]:
full_adder = ART / "full_adder.mlir"
print(pud.primitive_table(names=("row_add",)).split("\n\n")[0])
print(pud.primitive_table(flags=(f"--pud-primitives={full_adder}",), names=("row_add",)))

A file cannot redefine what a primitive *means*. Every definition is run on a model of
the substrate against the primitive's function, and refused if it does not compute it.
Here is a "full adder" whose sum is the majority — wrong:

In [ ]:
%%writefile ../artifacts/wrong_adder.mlir
// Not a full adder: the "sum" is the carry again.
func.func @row_add(%a: !bits.row<1>, %b: !bits.row<1>, %cin: !bits.row<1> {pud.resident = "DCC1"})
    -> (!bits.row<1>, !bits.row<1> {pud.resident = "DCC1"}) {
  %cout = bits.maj %a, %b, %cin : !bits.row<1>, !bits.row<1>, !bits.row<1> -> !bits.row<1>
  return %cout, %cout : !bits.row<1>, !bits.row<1>
}

In [ ]:
wrong = ART / "wrong_adder.mlir"
try:
    pud.primitive_table(flags=(f"--pud-primitives={wrong}",))
except RuntimeError as e:
    print(next(l for l in str(e).splitlines() if "LLVM ERROR" in l))

## An approximate adder

Exact addition is a carry chain: 8 commands per bit on this substrate, because every bit
waits for the carry of the one below. Many applications tolerate small errors in the low
bits. A classic approximation, the **lower-part OR adder**, computes the low *K* bits
without any carry — each result bit is `a | b` — and adds the upper bits exactly, starting
from carry 0:

| bits | computed by | commands per bit |
|---|---|---|
| 0 … K−1 | the approximate cell, `row_add_approx` (built in: `a \| b`) | 4 |
| K … n−1 | the full adder, carry starting at 0 | 8 |

`--pud-approx-add-low-bits=K` does that for every plain addition in a kernel
(reductions and multiplications stay exact). For a 16-bit add, a tile then costs
`4K + 8(16 − K) + 1` commands instead of 129.

The cell is a primitive like any other, so its network can come from a file too. To
make the accuracy numbers below come from exactly what the compiler compiles, the cells
are files, including the built-in OR, and are used throughout. Three of them, for later:

In [ ]:
%%writefile ../artifacts/cell_or.mlir
// a | b: the majority of a, b and a row of ones. The built-in cell.
func.func @row_add_approx(%a: !bits.row<1>, %b: !bits.row<1>) -> !bits.row<1> {
  %w    = arith.constant 1 : i64
  %zero = bits.create_all_zero_row %w : i64 -> !bits.row<1>
  %one  = bits.create_all_one_row %w : i64 -> !bits.row<1>
  %s    = bits.maj %a, %b, %one : !bits.row<1>, !bits.row<1>, !bits.row<1> -> !bits.row<1>
  return %s : !bits.row<1>
}

In [ ]:
%%writefile ../artifacts/cell_xor.mlir
// a ^ b: (a | b) & !(a & b), the sum bit of a half adder.
func.func @row_add_approx(%a: !bits.row<1>, %b: !bits.row<1>) -> !bits.row<1> {
  %w    = arith.constant 1 : i64
  %zero = bits.create_all_zero_row %w : i64 -> !bits.row<1>
  %one  = bits.create_all_one_row %w : i64 -> !bits.row<1>
  %or   = bits.maj %a, %b, %one : !bits.row<1>, !bits.row<1>, !bits.row<1> -> !bits.row<1>
  %and  = bits.maj %a, %b, %zero : !bits.row<1>, !bits.row<1>, !bits.row<1> -> !bits.row<1>
  %nand = bits.not %and : !bits.row<1> -> !bits.row<1>
  %s    = bits.maj %or, %nand, %zero : !bits.row<1>, !bits.row<1>, !bits.row<1> -> !bits.row<1>
  return %s : !bits.row<1>
}

In [ ]:
%%writefile ../artifacts/cell_copy.mlir
// a: ignore b in the low bits altogether.
func.func @row_add_approx(%a: !bits.row<1>, %b: !bits.row<1>) -> !bits.row<1> {
  %w    = arith.constant 1 : i64
  %zero = bits.create_all_zero_row %w : i64 -> !bits.row<1>
  %one  = bits.create_all_one_row %w : i64 -> !bits.row<1>
  return %a : !bits.row<1>
}

In [ ]:
cells = {name: ART / f"cell_{name}.mlir" for name in ("or", "xor", "copy")}
print(pud.primitive_table(flags=(f"--pud-primitives={cells['or']}",), names=("row_add_approx",)))

## Cost and accuracy as *K* grows

For each *K*, `vadd` is compiled at 32 banks and run on gem5. Its accuracy comes from
evaluating the same cell file in Python over 200,000 random pairs of 15-bit operands
(so the exact sum still fits in 16 bits), because reading a compiled kernel's results
back is not possible yet (see the end of this notebook).

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
a = rng.integers(0, 1 << 15, 200_000, dtype=np.int64)
b = rng.integers(0, 1 << 15, 200_000, dtype=np.int64)
exact = a + b

def accuracy(low_bits, cell_path):
    cell = pud.network_function(cell_path, "row_add_approx")
    got = pud.approx_add(a, b, 16, low_bits, cell)
    err = np.abs(got - exact)
    return dict(error_rate=float(np.mean(err != 0)), mean_abs=float(err.mean()),
                max_abs=int(err.max()), rmse=float(np.sqrt(np.mean(err.astype(float) ** 2))),
                mred=float(np.mean(err / np.maximum(exact, 1))))

def run(low_bits, cell="or", banks=32):
    flags = (f"--pud-primitives={cells[cell]}", f"--pud-approx-add-low-bits={low_bits}")
    k = pud.compile_kernel("vadd_1048576", banks, flags=flags, tag=f"vadd_approx_{cell}_K{low_bits}")
    return k, pud.simulate(k.exe)

rows = []
for K in (0, 4, 8, 12):
    k, s = run(K)
    rows.append(dict(K=K, row_ops=s.row_ops, expected=128 * (4 * K + 8 * (16 - K) + 1),
                     us=s.makespan_us, **accuracy(K, cells["or"])))
    print(f"K={K:2} done")

In [ ]:
base = rows[0]["us"]
print(f"{'K':>3} {'row ops':>8} {'expected':>8} {'makespan us':>11} {'speed-up':>8} "
      f"{'wrong':>7} {'mean |err|':>10} {'RMSE':>8} {'max |err|':>9} {'MRED':>8}")
for r in rows:
    print(f"{r['K']:3} {r['row_ops']:8} {r['expected']:8} {r['us']:11.3f} {base / r['us']:8.2f} "
          f"{r['error_rate']:7.1%} {r['mean_abs']:10.2f} {r['rmse']:8.2f} {r['max_abs']:9} {r['mred']:8.2e}")

## What the approximate sums look like

A few sums at *K* = 8, the low byte set apart. In it each result bit is `a | b` of the
operands' bits, so it goes wrong where both operands have a 1 (the true result is 0 there,
with a carry into the next bit) and wherever that carry would have reached. Above it the
adder is exact but starts from carry 0, so a carry out of the low byte is lost too: the
high byte can come out one smaller than it should (the fifth example). The error column is
approximate − exact.

In [ ]:
K = 8
cell = pud.network_function(cells["or"], "row_add_approx")
examples = [(0x1234, 0x0F0F), (0x00FF, 0x0001), (0x0A0A, 0x0505), (0x3C33, 0x03CC)]
examples += [(int(x), int(y)) for x, y in zip(a[:3], b[:3])]

def split(v):  # 16 bits, the K approximated low bits after the gap
    bits = f"{v:016b}"
    return bits[:16 - K] + " " + bits[16 - K:]

print(f"{'':>6} {'a':>17} {'b':>17} {'exact a + b':>17} {'approximate':>17} {'error':>6}")
for x, y in examples:
    exact_sum = x + y
    approx = int(pud.approx_add(x, y, 16, K, cell))
    print(f"{'':>6} {split(x)} {split(y)} {split(exact_sum)} {split(approx)} {approx - exact_sum:6}")
print(f"\nRMSE over {len(a):,} random pairs at K = {K}: {accuracy(K, cells['or'])['rmse']:.1f}")

`0x00FF + 0x0001` gets every bit of the low byte wrong and loses the carry into bit 8,
yet it is off by only 1: how many bits are wrong and how large the error is are different
things, which is why the tables report both. `0x0A0A + 0x0505` and `0x3C33 + 0x03CC` have
no two ones in the same place, so nothing needs a carry and the approximation is exact.
And it never overshoots: `a | b` is never more than `a + b`, bit by bit, so every error is
negative or zero.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot([r["rmse"] for r in rows[1:]], [base / r["us"] for r in rows[1:]], "o-")
for r in rows[1:]:
    ax.annotate(f"K={r['K']}", (r["rmse"], base / r["us"]), textcoords="offset points", xytext=(5, -12))
ax.set_xscale("log"); ax.set_xlabel("RMSE of the sum"); ax.set_ylabel("speed-up over exact")
ax.set_title("vadd, lower-part OR adder, 32 banks"); plt.tight_layout()

**What to look for.**

- The command count gem5 measures is exactly `128 × (4K + 8(16 − K) + 1)`: the mapping
  priced the approximate cell, and the program issued it.
- Each approximate bit saves four commands, so the makespan falls roughly linearly in *K*.
- The error grows by about a factor of 16 for every four more approximate bits (RMSE 5.5,
  90, 1447 at *K* = 4, 8, 12): an error in bit *K − 1* is worth 2^(K−1).

## Which cell?

The OR cell is only one choice. With the file you can try others — here at *K* = 8:
`a ^ b`, the sum bit of a half adder, and simply copying `a`, ignoring `b` in the low
bits.

In [ ]:
cmp = []
for name in ("or", "xor", "copy"):
    table = pud.primitive_table(flags=(f"--pud-primitives={cells[name]}",), names=("row_add_approx",))
    per_bit = int(table.splitlines()[1].split()[1])
    k, s = run(8, name)
    cmp.append(dict(cell=name, per_bit=per_bit, row_ops=s.row_ops, us=s.makespan_us, **accuracy(8, cells[name])))
print(f"{'cell':>5} {'cmds/bit':>8} {'row ops':>8} {'makespan us':>11} {'speed-up':>8} {'wrong':>7} {'RMSE':>8} {'MRED':>8}")
for r in cmp:
    print(f"{r['cell']:>5} {r['per_bit']:8} {r['row_ops']:8} {r['us']:11.3f} {base / r['us']:8.2f} "
          f"{r['error_rate']:7.1%} {r['rmse']:8.2f} {r['mred']:8.2e}")

**XOR is worse on both counts.** It takes as many commands as the full adder, so it
saves nothing, and its error is twice OR's (RMSE 181 against 90). Where both bits are 1 the true two-bit sum is
`10`: OR answers `1`, off by one, while XOR answers `0`, off by two. Where the bits differ
both are right.

**The copy is the cheapest cell,** one command a bit, and the fastest; its relative error
is about XOR's and its RMSE a little lower. So among these three, OR is the most accurate for its cost, and the copy is what to
reach for when speed matters more — the kind of trade-off a primitive file lets you explore
without touching the compiler. An approximation only pays if its cell is cheaper than the
full adder it replaces.

> **Try it:** copy one of the `%%writefile` cells above, write your own network into it
> (any `bits.maj` / `bits.not` over `%a`, `%b`, `%zero` and `%one`), add its name to
> `cells`, and compare it. The compiler proves the sequence it emits computes your
> network.

## What this does and does not show

The command counts and times are measured: gem5 ran the compiled programs. The command
sequence for each cell is proved by the compiler to compute that cell's network, and gem5
now carries out what the commands do to the rows. The **accuracy**, though, is computed in
Python from the same network, not read back from gem5: loading a kernel's data into the
bit-sliced layout and reading results out is the open step described in `HANDOVER.md`.